# Preprocess EDA

前処理の比較検討用ノートブック。

## データ読み込みと前処理関数

`train.csv` を読み込み、`raw / SNV / 1次微分 / 2次微分` を共通フォーマットで作成します。

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA

px.defaults.template = "plotly_white"


def find_project_root(start: Path) -> Path:
    """Return repository root by searching for pyproject.toml."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.toml が見つかりません。")



def spectral_columns(frame: pd.DataFrame) -> list[str]:
    """Return spectral column names excluding metadata columns."""
    metadata_cols = {"sample number", "species number", "樹種", "含水率"}
    return [col for col in frame.columns if col not in metadata_cols]



def snv_transform(matrix: np.ndarray) -> np.ndarray:
    """Apply Standard Normal Variate transform per spectrum."""
    mean = matrix.mean(axis=1, keepdims=True)
    std = matrix.std(axis=1, keepdims=True)
    std = np.where(std == 0, 1.0, std)
    return (matrix - mean) / std



def first_derivative(matrix: np.ndarray, wavenumbers: np.ndarray) -> np.ndarray:
    """Compute first derivative along spectral axis."""
    return np.gradient(matrix, wavenumbers, axis=1)



def second_derivative(matrix: np.ndarray, wavenumbers: np.ndarray) -> np.ndarray:
    """Compute second derivative along spectral axis."""
    first = first_derivative(matrix, wavenumbers)
    return np.gradient(first, wavenumbers, axis=1)


PROJECT_ROOT = find_project_root(Path.cwd())
TRAIN_PATH = PROJECT_ROOT / "data" / "train.csv"

train_df = pd.read_csv(TRAIN_PATH, encoding="cp932")
spectral_cols = spectral_columns(train_df)
wavenumbers = np.asarray(spectral_cols, dtype=float)

x_raw = train_df[spectral_cols].to_numpy(dtype=float)
preprocessed = {
    "raw": x_raw,
    "SNV": snv_transform(x_raw),
    "1次微分": first_derivative(x_raw, wavenumbers),
    "2次微分": second_derivative(x_raw, wavenumbers),
}

print(f"train shape: {x_raw.shape}")
print(f"preprocess keys: {list(preprocessed.keys())}")

## 1. raw / SNV / 1次微分 / 2次微分 の比較

含水率レンジに沿って代表サンプルを抽出し、4種類の前処理でスペクトル形状がどう変わるかを同時に比較します。

In [ ]:
n_samples = 10
sorted_idx = np.argsort(train_df["含水率"].to_numpy())
selected_positions = np.linspace(0, len(sorted_idx) - 1, n_samples, dtype=int)
selected_idx = sorted_idx[selected_positions]

subset_meta = train_df.iloc[selected_idx][["sample number", "樹種", "含水率"]].reset_index(drop=True)

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=["raw", "SNV", "1次微分", "2次微分"],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

plot_order = ["raw", "SNV", "1次微分", "2次微分"]
for i, key in enumerate(plot_order):
    row = i // 2 + 1
    col = i % 2 + 1
    x_mat = preprocessed[key]
    for j, original_idx in enumerate(selected_idx):
        meta = train_df.iloc[original_idx]
        fig.add_trace(
            go.Scatter(
                x=wavenumbers,
                y=x_mat[original_idx],
                mode="lines",
                line=dict(width=1.6),
                name=f"sample {int(meta['sample number'])}",
                legendgroup=str(int(meta["sample number"])),
                showlegend=(i == 0),
                customdata=np.column_stack(
                    [
                        np.repeat(int(meta["sample number"]), len(wavenumbers)),
                        np.repeat(meta["樹種"], len(wavenumbers)),
                        np.repeat(float(meta["含水率"]), len(wavenumbers)),
                    ]
                ),
                hovertemplate=(
                    "sample=%{customdata[0]}<br>"
                    "樹種=%{customdata[1]}<br>"
                    "含水率=%{customdata[2]:.2f}<br>"
                    "波数=%{x:.1f}<br>"
                    "値=%{y:.4f}<extra></extra>"
                ),
            ),
            row=row,
            col=col,
        )

fig.update_xaxes(autorange="reversed", title="波数 (cm^-1)", showgrid=True)
fig.update_yaxes(showgrid=True)
fig.update_layout(
    title="前処理ごとのスペクトル比較 (代表サンプル)",
    height=900,
    hovermode="x unified",
    legend_title_text="sample number",
    margin=dict(t=100, r=20, b=40, l=60),
)
fig.show()

subset_meta

## 2. PCAスコアプロット

各前処理で `PCA(2成分)` を行い、`PC1-PC2` 空間でサンプルの分布を比較します。色は含水率です。

In [ ]:
score_frames: list[pd.DataFrame] = []
explained_ratio: dict[str, tuple[float, float]] = {}

for key, x_mat in preprocessed.items():
    pca = PCA(n_components=2)
    scores = pca.fit_transform(x_mat)
    explained_ratio[key] = (float(pca.explained_variance_ratio_[0]), float(pca.explained_variance_ratio_[1]))
    score_frames.append(
        pd.DataFrame(
            {
                "前処理": key,
                "PC1": scores[:, 0],
                "PC2": scores[:, 1],
                "含水率": train_df["含水率"].to_numpy(),
                "樹種": train_df["樹種"].to_numpy(),
                "sample number": train_df["sample number"].to_numpy(),
            }
        )
    )

scores_df = pd.concat(score_frames, ignore_index=True)
fig = px.scatter(
    scores_df,
    x="PC1",
    y="PC2",
    color="含水率",
    facet_col="前処理",
    facet_col_wrap=2,
    color_continuous_scale="Viridis",
    height=900,
    hover_data=["sample number", "樹種"],
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_traces(marker=dict(size=5, opacity=0.75))
fig.update_layout(
    title="前処理ごとの PCA スコアプロット (PC1 vs PC2)",
    margin=dict(t=80, r=20, b=40, l=60),
)
fig.show()

pd.DataFrame(
    {
        "前処理": list(explained_ratio.keys()),
        "PC1寄与率": [v[0] for v in explained_ratio.values()],
        "PC2寄与率": [v[1] for v in explained_ratio.values()],
    }
).round(4)

## 3. PCAローディング

各前処理で `PC1` と `PC2` のローディングを波数軸で確認します。どの波数帯が主成分に効いているかを比較できます。

In [ ]:
loading_fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=["raw", "SNV", "1次微分", "2次微分"],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

for i, (key, x_mat) in enumerate(preprocessed.items()):
    pca = PCA(n_components=2)
    pca.fit(x_mat)
    row = i // 2 + 1
    col = i % 2 + 1

    loading_fig.add_trace(
        go.Scatter(
            x=wavenumbers,
            y=pca.components_[0],
            mode="lines",
            line=dict(color="#636EFA", width=2),
            name="PC1",
            legendgroup="PC1",
            showlegend=(i == 0),
            hovertemplate="波数=%{x:.1f}<br>PC1 loading=%{y:.5f}<extra></extra>",
        ),
        row=row,
        col=col,
    )
    loading_fig.add_trace(
        go.Scatter(
            x=wavenumbers,
            y=pca.components_[1],
            mode="lines",
            line=dict(color="#EF553B", width=2),
            name="PC2",
            legendgroup="PC2",
            showlegend=(i == 0),
            hovertemplate="波数=%{x:.1f}<br>PC2 loading=%{y:.5f}<extra></extra>",
        ),
        row=row,
        col=col,
    )

loading_fig.update_xaxes(autorange="reversed", title="波数 (cm^-1)", showgrid=True)
loading_fig.update_yaxes(title="loading", showgrid=True, zeroline=True)
loading_fig.update_layout(
    title="前処理ごとの PCA ローディング (PC1 / PC2)",
    height=900,
    hovermode="x unified",
    legend_title_text="",
    margin=dict(t=100, r=20, b=40, l=60),
)
loading_fig.show()